# Nearfield Potential Evaluation

Potential evaluation after the BEM solve asks for the field away from, close to, or on the boundary:

$$
u(x)=\sum_{T\subset\Gamma}\int_T G(x,y)\,\rho_h(y)\,d\sigma(y).
$$

For target points close to a source element, the kernel is nearly singular. Fixed order quadrature and pure farfield expansions lose accuracy exactly where the visualized potential is often most interesting.

## Local Geometry

For a close triangular source element, NGSolve projects the target point onto the reference triangle and builds a tangent triangle at the projected point:

$$
\xi_0=\operatorname*{arg\,min}_{\xi\in\widehat{T}} |x-F_T(\xi)|,
\qquad
p_0=F_T(\xi_0),
\qquad
F_{T,\mathrm{flat}}(\xi)=p_0+J_T(\xi_0)(\xi-\xi_0).
$$

![Nearfield geometry](images/nearfield_triangle.png)

The singular correction is computed on $T_{\mathrm{flat}}$, while the usual element evaluator supplies the local source value at $\xi_0$.

## Analytical Correction

On a near element $T$, split the evaluation into the already computed base contribution and a local singular correction:

$$
u_T(x)
\approx
u_T^{\mathrm{base}}(x)
+
\rho_h(p_0)\,
\left(
I_T^{\mathrm{analytic}}(x)-I_T^{\mathrm{quad}}(x)
\right).
$$

Here

$$
I_T^{\mathrm{analytic}}(x)=\int_{T_{\mathrm{flat}}}\frac{1}{4\pi |x-y|}\,d\sigma(y).
$$

The analytical term is evaluated by closed triangle formulas. The term $I_T^{\mathrm{quad}}$ is the same singular part evaluated by the standard flat quadrature rule. This replaces only the poorly resolved singular part and avoids double counting.

## Supported Singular Parts

The implemented analytical triangle formulas are

$$
\int_T \frac{1}{4\pi |x-y|}\,d\sigma(y),
\qquad
\int_T \nabla_x\left(\frac{1}{4\pi |x-y|}\right)\,d\sigma(y),
\qquad
\int_T \frac{\partial}{\partial n_y}\left(\frac{1}{4\pi |x-y|}\right)\,d\sigma(y).
$$

They are used for Laplace single layer, the gradient of the Laplace single layer, and Laplace double layer. Helmholtz single and double layer potentials use the same Laplace singular correction, because the factor $e^{i\kappa |x-y|}$ is smooth at $x=y$; the singular behavior is still the Laplace $1/|x-y|$ part.

## Local Expansion Path

For many target points in a region, the potential can first be represented by a local expansion:

$$
u^{\mathrm{base}}(x)=\mathcal L_{\mathcal R}(x),
\qquad x\in\mathcal R.
$$

Near source elements are then corrected pointwise:

$$
u(x)=\mathcal L_{\mathcal R}(x)
+\sum_{T\in\mathcal N(x)}
\rho_h(p_{0,T})
\left(I_T^{\mathrm{analytic}}(x)-I_T^{\mathrm{quad}}(x)\right).
$$

If no analytical triangle formula is registered for the kernel, the fallback is a numerical correction: subtract the standard element quadrature and add a point adapted Duffy rule on the close triangle.